## 1. Introduction

This notebook documents the full modeling workflow used to classify country-level epidemic risk into low, moderate, high, and critical categories. We use a Random Forest classifier because it performs strongly on structured tabular epidemiological features and can capture nonlinear interactions without requiring extensive scaling assumptions. In public-health forecasting settings, interpretability is essential, so we pair model training with SHAP analysis to quantify feature contributions. This combination allows us to validate predictive utility while also checking biological plausibility of learned relationships. The final objective is not only high classification performance but also transparent decision support for outbreak surveillance.


In [ ]:
from __future__ import annotations

from pathlib import Path
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("epiwatch-modeling")

sns.set_theme(style="whitegrid")
ROOT = Path("..")
DATA_DIR = ROOT / "data"

PROCESSED_PATH = DATA_DIR / "processed.csv"
SHAP_PATH = DATA_DIR / "shap_values.csv"


## 2. Dataset Loading and Feature Statistics

We begin by loading the processed modeling dataset and checking basic descriptive statistics for key epidemiological features. This step is essential for identifying scale mismatches, missingness, and potential data-quality issues before model fitting. We summarize mean, standard deviation, minimum, and maximum values to understand the dynamic range of each predictor. These summary statistics also provide context for interpreting feature importance magnitudes later in the notebook.


In [ ]:
df = pd.read_csv(PROCESSED_PATH)
logger.info("Loaded processed dataset with shape: %s", df.shape)

feature_candidates = [
    "rt_estimate",
    "case_growth_rate",
    "rolling_7d_avg",
    "vax_coverage",
    "stringency_index",
    "risk_score",
]
features = [c for c in feature_candidates if c in df.columns]

if "risk_category" not in df.columns:
    bins = [0, 25, 50, 75, 100]
    labels = ["low", "moderate", "high", "critical"]
    df["risk_category"] = pd.cut(
        pd.to_numeric(df.get("risk_score", 0), errors="coerce").fillna(0),
        bins=bins,
        labels=labels,
        include_lowest=True,
    ).astype(str)

stats = (
    df[features]
    .apply(pd.to_numeric, errors="coerce")
    .agg(["mean", "std", "min", "max"])
    .T
    .round(4)
)
stats


## 3. Class Balance

Class balance directly affects model calibration and error distribution across risk categories. We inspect class frequencies to determine whether the classifier is likely to overfit dominant classes or underperform on rarer high-severity outcomes. In epidemiological settings, minority classes often carry the highest operational cost, so imbalance awareness is critical. The bar chart below gives a quick diagnostic of label prevalence and helps motivate downstream evaluation choices.


In [ ]:
class_counts = df["risk_category"].astype(str).value_counts().sort_index()
class_counts

plt.figure(figsize=(8, 4))
ax = sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
ax.set_title("Risk Category Distribution")
ax.set_xlabel("Risk Category")
ax.set_ylabel("Count")
for i, v in enumerate(class_counts.values):
    ax.text(i, v + max(class_counts.values) * 0.01, str(v), ha="center", fontsize=10)
plt.tight_layout()
plt.show()


## 4. Feature Engineering Validation

The estimated reproduction number (Rt) should behave like a plausible transmission signal rather than a noisy artifact. In most stable epidemic periods, values should cluster around 1.0, with right-tail mass corresponding to active growth phases. We visualize the Rt distribution to verify this expected shape and inspect central tendency against epidemiological intuition. This check helps validate that upstream feature engineering is producing biologically interpretable inputs.


In [ ]:
rt = pd.to_numeric(df["rt_estimate"], errors="coerce").dropna()

plt.figure(figsize=(8, 4))
sns.histplot(rt, bins=50, kde=True, color="#22c55e")
plt.axvline(1.0, color="#ef4444", linestyle="--", linewidth=1.5, label="Rt = 1.0")
plt.title("Distribution of Rt Estimates")
plt.xlabel("Rt estimate")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Rt mean: {rt.mean():.3f}")
print(f"Rt median: {rt.median():.3f}")


## 5. Model Training

We train a Random Forest classifier on engineered epidemiological features and evaluate on a stratified hold-out split. Random Forest is robust to nonlinear interactions and heterogeneous feature distributions, which is useful for cross-country surveillance data. During training, we prioritize reproducibility with fixed random seeds and report the primary metric (accuracy) for a first-pass performance check. This section establishes the core predictive baseline used for subsequent error analysis and interpretation.


In [ ]:
X = df[features].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True)).fillna(0)
y = df["risk_category"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None,
)

logger.info("Training RandomForestClassifier...")
rf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f} ({acc * 100:.2f}%)")


## 6. Confusion Matrix

A confusion matrix provides class-wise error structure that aggregate accuracy can hide. We compute a normalized matrix to compare misclassification rates across classes even when class support differs. This makes it easier to identify whether severe categories are being confused with adjacent states. In applied outbreak triage, these directional errors matter because underestimating severity can delay interventions. The normalized heatmap below gives a direct view of per-class recall behavior.


In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    cmap="mako",
    xticklabels=labels,
    yticklabels=labels,
)
plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## 7. ROC Curves

To evaluate ranking quality beyond hard labels, we compute one-vs-rest ROC curves for each risk category. This analysis measures how well the model separates each class from all others over all possible thresholds. Plotting all curves together allows direct comparison of class separability and highlights any weak category boundaries. For public-health deployment, strong ROC behavior across all classes increases confidence in threshold-based alerting policies.


In [ ]:
proba = rf.predict_proba(X_test)
classes = rf.classes_
y_test_bin = label_binarize(y_test, classes=classes)

plt.figure(figsize=(8, 6))
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.7)
plt.title("One-vs-Rest ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 8. SHAP Summary

SHAP values provide additive local explanations that can be aggregated to understand global model behavior. We load precomputed SHAP values and render a beeswarm-style summary to visualize both magnitude and direction of feature effects. This plot reveals which variables dominate prediction shifts and whether higher feature values increase or decrease risk. In an epidemiological tool, this is crucial for validating that learned patterns align with mechanistic expectations.


In [ ]:
shap_df = pd.read_csv(SHAP_PATH)
shap_features = [c for c in shap_df.columns if c in X.columns]

X_for_shap = X[shap_features].iloc[: len(shap_df)].copy()
shap_values_matrix = shap_df[shap_features].to_numpy()

shap_explanation = shap.Explanation(
    values=shap_values_matrix,
    data=X_for_shap.to_numpy(),
    feature_names=shap_features,
)

shap.plots.beeswarm(shap_explanation, max_display=10)


## 9. SHAP Dependence

Dependence plots help inspect how marginal changes in a single feature translate into SHAP contribution shifts. Because Rt is epidemiologically central, we analyze its dependence behavior explicitly to see whether the model responds smoothly around the epidemic threshold. We expect increasing Rt to generally push SHAP contributions upward toward higher risk classes, especially above values near 1.0. This section provides a direct interpretability check connecting model internals to transmission dynamics theory.


In [ ]:
target_feature = "rt_estimate" if "rt_estimate" in shap_features else shap_features[0]
shap.dependence_plot(
    target_feature,
    shap_values_matrix,
    X_for_shap,
    interaction_index=None,
    alpha=0.6,
)
plt.tight_layout()
plt.show()


## 10. Conclusions

The SHAP analysis indicates that risk-related transmission features drive classification behavior in a way that is epidemiologically coherent. Variables tied to transmission intensity and momentum, particularly Rt-associated signals, exert the largest contributions to predicted risk movement. This supports biological plausibility because changes in effective reproduction should precede and explain shifts in outbreak severity classes. The confusion matrix and ROC diagnostics suggest that class boundaries are generally well separated, with strongest discrimination where signal strength is highest. Overall, the model appears both performant and interpretable enough to support real-time surveillance workflows, provided that data quality and reporting lags are continually monitored.
